<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/05_Visualising_Networks/04_nxpandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕸️ Co-abundance networks — a pandas and NetworkX practical

**Day 3 · 10:30–11:00 · Multi-omics Data Science**

---

An hour ago we learned what a graph is and how NetworkX represents one. Now we build a
network out of real measurements — the same 45 septic patients we have followed since
Monday — using nothing but pandas and a correlation coefficient. Two proteins become
neighbours when their serum abundances rise and fall together across the cohort.

That construction is seductive, so we say the hard part first. **A correlation is not an
interaction.** It says the two proteins move together in *these* patients; it says nothing
about whether they touch, and nothing about which one moves first. Both may simply be
downstream of the same driver — how ill the patient is, how old they are, or the
acute-phase response that dominates any serum proteome. Every picture we draw today is a
hypothesis generator, and the hypotheses are only as good as the choices behind them.

The second half of the session is plumbing, and the plumbing matters: at 11:00 we move to
**Cytoscape**, and Cytoscape needs a node table and an edge table. Building those two
tables carefully — with the gene symbols, the network measures and the published fold
changes already attached — is what separates a figure you can publish from a screenshot
you cannot.

### What you will be able to do afterwards

1. Take an omics matrix in pandas and turn it into a correlation matrix, an edge list and
   a NetworkX graph, understanding what each transposition and filter is for.
2. Choose a correlation threshold deliberately, and say out loud what your network would
   have looked like had you chosen differently.
3. Rank nodes by degree, betweenness and closeness — and explain why a hub in a
   co-abundance network is as often an artefact as a discovery.
4. Export a network with its node attributes to Cytoscape by two independent routes, and
   know which one to trust.

## 🧬 What an edge means, and what it does not

A co-abundance edge is a statement about a table, not about a cell. We compute, for every
pair of proteins, the correlation of their log₂ abundances across 45 serum samples, and we
draw an edge when that correlation is large. Nothing in that procedure knows about binding
sites, complexes or pathways.

There are at least four reasons two serum proteins correlate, and only one of them is
interesting:

| Reason | What the edge means | How common in serum |
|---|---|---|
| **Shared upstream driver** | both respond to the same signal — infection severity, the acute-phase response, age, renal function | very common, and usually the dominant one |
| **Same tissue of origin** | both leak into blood when the same organ is damaged | common in sepsis |
| **Technical coupling** | shared peptides between protein groups, the same normalisation, the same missingness pattern | common, and easy to miss |
| **Direct biology** | the two really do act together — a complex, an enzyme and its regulator | the one you are hoping for |

The network cannot tell these apart. What it can do is give you a **short, structured list
of candidates** to check against interaction databases, pathway annotation and the
literature — which is exactly what a Cytoscape session is for, and what Multi-omics II does
this afternoon with two omics layers at once.

> ⚙️ If you want physical interactions, do not infer them: look them up. STRING, IntAct
> and Reactome exist. A correlation network and an interaction network render identically
> on screen, and confusing them is the most common way a network figure misleads a
> reader.

### The route through the notebook

Nine steps, each one a decision rather than a formality. Where a step has a knob, we say
what turning it does.

| Step | What we do | The decision hiding in it |
|---|---|---|
| 1–2 | read the course data with pandas and look at it | which comparison, which columns |
| 3 | correlate proteins across patients | Pearson or Spearman; which proteins to keep |
| 4 | stack the matrix into an edge list | the threshold — the single biggest choice |
| 5 | build the graph | what to carry onto the edges and nodes |
| 6 | draw it | layout, colour, and what colour is allowed to mean |
| 7 | rank nodes by centrality | whether a hub is biology or abundance |
| 8–9 | annotate and export | what a colleague needs to reproduce your figure |

## 1. Load the data

We use the course dataset throughout — the same serum proteomics that Day 2 afternoon put
through differential abundance, and the same 45 patients that this afternoon's integration
session pairs with the metabolome. Nothing here is a toy example; the network we build is
built on the numbers the paper was built on.

Three files matter for this notebook, all of them in the course repository:

| File | Content |
| --- | --- |
| `proteomics/data/protein_groups_matrix.tsv` | MaxLFQ protein intensities: 4 annotation columns (`protein_group`, `protein_names`, `genes`, `description`), then 45 patient samples (`Con1`–`Con15`, `KP1`–`KP15`, `CRKP1`–`CRKP15`) and 3 pooled quality-control runs (`QC_pool1`–`QC_pool3`) |
| `metadata/sample_metadata.tsv` | One row per sample: `sample_id`, `group`, `group_order`, `age`, `sex` and clinical laboratory values |
| `proteomics/data/published_deps.tsv` | Differentially expressed proteins reported in the paper, per `comparison` (`CRKP_vs_KP`, `CRKP_vs_Con`, `KP_vs_Con`, `CRKP_vs_KP_vs_Con`), with `log2_fold_change` and `p_value` |

The source is He J *et al.* (2026) *Serum proteomic profiling of sepsis patients reveals a
protein-based diagnostic model, with metabolomic insights into carbapenem-resistant*
Klebsiella pneumoniae *infection*, Front Immunol 17:1818068.

### The cohort behind the matrix

Serum was collected on the day of diagnosis (day 0) from 45 patients with sepsis, 15 per
group:

- `Con` — sepsis with negative blood cultures;
- `CSKP` — sepsis caused by carbapenem-**susceptible** *Klebsiella pneumoniae* (the sample
  identifiers of this group start with `KP`);
- `CRKP` — sepsis caused by carbapenem-**resistant** *K. pneumoniae*.

The samples were measured by diaPASEF on a timsTOF Pro and processed with DIA-NN, giving
MaxLFQ intensities for **1 458 protein groups** with about **27 % missing values**.

The clinical question is a practical one: **can host molecules in serum tell a resistant
infection apart from a susceptible one on day 0**, before the culture and antibiogram come
back two or three days later? Antibiotic choice on day 0 changes outcomes, so a
host-response signature available immediately would be useful.

This notebook does not test that question — Day 2 afternoon did, protein by protein. We
ask a structural question about the same matrix instead: **which proteins move together
across these 45 patients**, and what shape does that make? Keep the two questions apart in
your head. A differential protein is a difference between groups; a network edge is a
covariation within the cohort. A protein can be one without being the other.

> 🧬 Every patient here is septic. That is easy to forget and it changes everything: the
> variation we are about to correlate is variation *among* very sick people, dominated by
> how sick each one is. The pathogen's resistance status is a small perturbation on top of
> a very large acute-phase signal.

### Reading data with pandas

`pd.read_csv` reads any delimited text file; for a tab-separated file we pass `sep="\t"`.
The helper below is a small convenience that keeps the notebook runnable in two places at
once: if you have cloned the course repository it reads the local file, and on Colab it
falls back to the raw GitHub URL. Fetching data at run time rather than shipping it beside
the notebook is a habit worth copying — there is then exactly one authoritative copy.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"


def data_path(relative_path):
    """Locate a course data file.

    If the file exists in a local clone of the course repository (searching the
    current folder and its parents) we use that copy; otherwise we fall back to
    reading it straight from GitHub, which is what happens on Google Colab.
    """
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        candidate = folder / relative_path
        if candidate.exists():
            return str(candidate)
    return f"{BASE_URL}/{relative_path}"


PROTEIN_MATRIX = data_path("proteomics/data/protein_groups_matrix.tsv")
SAMPLE_METADATA = data_path("metadata/sample_metadata.tsv")
PUBLISHED_DEPS = data_path("proteomics/data/published_deps.tsv")

print("protein matrix :", PROTEIN_MATRIX)
print("sample metadata:", SAMPLE_METADATA)
print("published DEPs :", PUBLISHED_DEPS)

In [ ]:
# pandas reads a tab separated file with read_csv and sep="\t"
deps = pd.read_csv(PUBLISHED_DEPS, sep="\t")
deps.head()

## 2. Exploratory analysis

Before computing anything, look at what you loaded. The point of the next few cells is not
the syntax — it is the reflex. Print the shape, read the column names, check that the
categories are the ones you expect and that the numbers are on the scale you assumed. Most
analysis bugs are visible in the first ten rows and invisible thereafter.

We start with the **published differential proteins**, the paper's own results table, which
we will come back to in section 8 to annotate our network nodes.

**The first rows.** `head()` returns the first five rows by default, or as many as you ask
for. It is the cheapest sanity check there is.

In [ ]:
deps.head(10)

**The last rows.** `tail()` is the mirror image, and worth the keystroke: files that were
concatenated badly, or that carry a stray total row at the bottom, only give themselves
away at the end.

In [ ]:
deps.tail(10)

**Summary statistics.** `describe()` reports count, mean, standard deviation and quartiles
for every numeric column, and silently skips the text ones. Read the `count` row first —
where it falls below the number of rows, that column has missing values.

In [ ]:
deps.describe()

**The columns, and what is in them.** `deps.columns.tolist()` names the fields;
`value_counts()` on `comparison` tells us how many proteins the paper reported for each
contrast. This table stacks several comparisons on top of each other, so *filtering by
`comparison` before doing anything else* is not optional — otherwise the same protein
appears more than once with different fold changes.

In [ ]:
print(deps.columns.tolist())
print()
print(deps["comparison"].value_counts())

### Selecting rows, then dropping columns

Two different operations, easy to confuse. **Selecting rows** with a boolean mask
(`deps[deps["comparison"] == "CRKP_vs_KP"]`) narrows the table to the comparison we care
about most: both groups have a confirmed *K. pneumoniae* infection and differ only in
resistance, so anything that separates them is about resistance rather than about being
infected. **Dropping columns** with
[`drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop.html)
and `axis=1` removes the fields we will not use.

`axis=1` means columns; `axis=0` means rows. Getting that argument wrong is a rite of
passage, and pandas will happily let you do it.

In [ ]:
# keep only the comparison we care about most: resistant vs susceptible K. pneumoniae
crkp_vs_kp = deps[deps["comparison"] == "CRKP_vs_KP"]

# drop the columns we do not need (axis=1 means "columns", not "rows")
crkp_vs_kp = crkp_vs_kp.drop(
    ["comparison", "mean_case", "mean_control", "fold_change"], axis=1
)

print(crkp_vs_kp.shape)
crkp_vs_kp.sort_values("log2_fold_change").head()

## 3. A protein–protein correlation matrix

Here is the step that creates the network. Everything after it is bookkeeping.

`corr` correlates the **columns** of a data frame. Our matrix has one row per protein group
and one column per sample, so to obtain a **protein–protein** correlation matrix we set
`protein_group` as the index and **transpose** before calling
[`corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html). If you
forget the transpose you get a perfectly valid 45 × 45 patient similarity matrix, which is
a different and also interesting object — it is what Similarity Network Fusion works on
this afternoon — but it is not what we want here.

Two things to settle first:

1. **Separate the annotation and QC columns from the 45 patient columns.** The `QC_pool*`
   runs are the same pooled sample injected repeatedly to monitor instrument drift. They
   are not biological replicates, and letting them into a correlation across patients would
   add three points that share no biology with the cohort.
2. **Work on log₂ intensities.** MaxLFQ values span orders of magnitude and are
   multiplicative; on the log scale the distribution is roughly symmetric, which is what a
   Pearson correlation assumes.

### ⚙️ What n = 45 buys you, and what it does not

A correlation coefficient estimated from 45 patients is a noisy thing, and it helps to know
how noisy. With n = 45, the smallest correlation that reaches nominal significance at
p < 0.05 is **|r| ≈ 0.29** — so if you thresholded there, pure sampling noise would hand
you a dense and entirely fictional network. Our threshold of 0.7 corresponds to
p ≈ 9 × 10⁻⁸, which is a very different proposition.

But the number of tests is large. Correlating the 150 proteins we keep below means
**11 175 pairs**; correlating all 1 458 protein groups would mean over **a million**. At
nominal p < 0.05 across 11 175 independent noise pairs you would expect around 560 spurious
edges. Two honest qualifications follow. First, the pairs are *not* independent — proteins
driven by the same signal correlate with each other, so neither the naive multiple-testing
arithmetic nor a Bonferroni correction is quite right. Second, and more important, a
correlation can be real, highly significant and still biologically uninformative if both
proteins are simply tracking severity.

> ⚙️ **Pearson or Spearman?** Pearson measures linear covariation and is sensitive to
> outliers and skew; a single extreme patient can manufacture r = 0.8 on its own. Spearman
> correlates the *ranks*, which costs a little power when the data really are Gaussian and
> buys a lot of robustness when they are not. Proteomics data are not clean and Gaussian —
> they are imputed, right-skewed at the low end and full of individual patients with
> extreme acute-phase values. Spearman is the safer default; we use Pearson here because
> the log₂ transform and the complete-cases filter below make it defensible, and because
> it lets you compare against `.corr(method="spearman")` as an exercise.

In [ ]:
proteins = pd.read_csv(PROTEIN_MATRIX, sep="\t")

annotation_columns = ["protein_group", "protein_names", "genes", "description"]
qc_columns = [c for c in proteins.columns if c.startswith("QC_pool")]
patient_columns = [
    c for c in proteins.columns if c not in annotation_columns + qc_columns
]

# annotation table: one row per protein group, indexed by protein_group
annotation = proteins[annotation_columns].set_index("protein_group")

# numeric matrix: proteins (rows) x patient samples (columns), on the log2 scale
intensities = proteins.set_index("protein_group")[patient_columns]
log_intensities = np.log2(intensities)

print(f"{proteins.shape[0]} protein groups")
print(f"{len(patient_columns)} patient samples, e.g. {patient_columns[:3]} ... {patient_columns[-3:]}")
print(f"{len(qc_columns)} QC pools kept aside: {qc_columns}")

# sanity check against the metadata: the 45 columns should be exactly the 45 samples
metadata = pd.read_csv(SAMPLE_METADATA, sep="\t").set_index("sample_id")
assert set(patient_columns) == set(metadata.index), "columns and metadata disagree"
print()
print(metadata["group"].value_counts().sort_index())

log_intensities.iloc[:5, :5]

### How complete is the matrix, and on what scale?

Two histograms, and both are worth a minute. The left one counts, for each protein group,
in how many of the 45 patients it was quantified — proteomics missingness is not random,
and a protein measured in 12 patients cannot enter a correlation across 45. The right one
shows the log₂ intensity distribution: roughly symmetric with a tail at the low end, which
is the shape a Pearson correlation is entitled to assume.

In [ ]:
print("matrix shape (proteins x samples):", log_intensities.shape)
print(f"missing values: {100 * log_intensities.isna().mean().mean():.1f}%")

# in how many of the 45 patients was each protein quantified?
observed_per_protein = log_intensities.notna().sum(axis=1)

values = log_intensities.to_numpy().ravel()
values = values[~np.isnan(values)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(observed_per_protein, bins=45, color="#4878A8")
axes[0].set_xlabel("number of samples with a value")
axes[0].set_ylabel("number of protein groups")
axes[0].set_title("Completeness per protein")
axes[1].hist(values, bins=60, color="#A85878")
axes[1].set_xlabel("log2 MaxLFQ intensity")
axes[1].set_ylabel("number of measurements")
axes[1].set_title("Intensity distribution")
fig.tight_layout()
plt.show()

print(f"{(observed_per_protein == len(patient_columns)).sum()} proteins are complete in all {len(patient_columns)} samples")

### Choosing which proteins go into the network

Now a decision, and it is a real one. We could correlate all 1 458 protein groups, but the
matrix would be over a million pairs, and most of it would be driven by the missing values
rather than by biology. So we cut twice:

- **complete cases only** — proteins quantified in all 45 patients. This avoids correlating
  two proteins on the handful of samples where both happen to be present, which is how you
  manufacture r = 0.95 from four points. The price is a bias towards **abundant** proteins,
  because abundant proteins are the ones without gaps. Remember that when we look at hubs.
- **the 150 most variable of those** — a protein whose abundance is the same in every
  patient carries no information about co-regulation, whatever its biological importance.

Both cuts are defensible and both are arbitrary in their exact value. The network you are
about to see is the network *of those 150 proteins*; a protein that did not make the cut
cannot appear in it, cannot be a hub, and cannot be found interesting. That is not a flaw
in the method — it is a fact about the method that you must state when you show the
figure.

In [ ]:
# Compute the correlation matrix.
#
# Keeping every protein would give a 1458 x 1458 matrix (over a million pairs),
# most of it driven by missing values. We therefore keep
#   (a) only proteins measured in ALL 45 patients, and
#   (b) among those, the 150 most variable ones - a protein that barely changes
#       between patients cannot tell us anything about co-regulation.
complete = log_intensities.dropna(axis=0)

variability = complete.std(axis=1).sort_values(ascending=False)
N_PROTEINS = 150
most_variable = complete.loc[variability.index[:N_PROTEINS]]

# transpose: samples become the rows, proteins the columns, so corr() gives us
# a protein x protein matrix of Pearson correlation coefficients
corr_matrix = most_variable.T.corr()
corr_matrix = corr_matrix.rename_axis(index="source", columns="target")

print("proteins measured in every sample:", complete.shape[0])
print("correlation matrix shape:", corr_matrix.shape)
corr_matrix.iloc[:5, :5].round(2)

## 4. From matrix to edge list

A NetworkX graph is built from a list of pairs, not from a square matrix, so we reshape.
[`stack`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html)
turns a wide frame into a long one: each cell of the matrix becomes a row with its row
label, its column label and its value — which is precisely the shape `source, target,
weight`.

A correlation matrix is symmetric and its diagonal is 1, since every protein correlates
perfectly with itself. We therefore keep only the **upper triangle** (`np.triu` with `k=1`
excludes the diagonal), so that each protein pair appears exactly once and no self-loops
are created. Skip this and you get a graph with twice the edges and a self-loop on every
node — which NetworkX will accept without complaint and which will quietly corrupt every
degree and every centrality afterwards.

In [ ]:
# keep the upper triangle only (k=1 excludes the diagonal)
upper_triangle = np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1)

# turn the data frame corr_matrix into an edge list
edges = corr_matrix.where(upper_triangle).stack().reset_index()
edges = edges.dropna()

print(f"{len(edges)} protein pairs")
edges.head()

`stack` names the columns after the index levels it unstacked, so we rename them to
`source`, `target` and `weight` — the names `nx.from_pandas_edgelist` expects, and the names
Cytoscape's import dialog recognises. Consistent column naming is a small thing that saves
a large amount of clicking later.

In [ ]:
edges.columns = ["source", "target", "weight"]
edges.head()

### ⚠️ The threshold decides the network

This is the most consequential line of code in the notebook, and it is a single number.

Every pair of proteins has *some* correlation. A network appears only when we declare a
cut-off and throw the rest away, and everything downstream inherits that declaration: the
number of edges, the density, whether the graph is one component or twenty, which
communities form, and every centrality ranking we compute in section 7. Lower the threshold
and the network thickens until it is a hairball in which everything is central; raise it
and the network shatters into disconnected pairs. Neither extreme is wrong; both are
choices.

So treat the threshold as a parameter to be reported and probed, not a constant to be
forgotten:

| Cut-off on absolute r | What you get | When it is reasonable |
|---|---|---|
| 0.5 | dense, most nodes connected, hubs everywhere | screening for anything at all; expect noise |
| 0.7 | our choice — sparse enough to read, still connected | n ≈ 45, exploratory network for follow-up |
| 0.9 | a few tight cliques, most nodes isolated | you want only the near-identical pairs |

**Sample size changes what a given number means.** With n = 45 patients, |r| = 0.7 is far
beyond what noise produces (p ≈ 9 × 10⁻⁸). With n = 15 — one group on its own — the same
0.7 has p ≈ 0.004, which across thousands of pairs is reached by chance routinely. A
per-group network at the same threshold is therefore *not* comparable to this one, and if
it looks denser that is an artefact of the smaller sample, not a biological finding.

We threshold on the **absolute** value because a negative correlation is as informative as
a positive one: two proteins that consistently move in opposite directions are linked too.
The sign is kept as an edge attribute and used later for edge colour.

In [ ]:
THRESHOLD = 0.7

strong_edges = edges[np.absolute(edges["weight"]) >= THRESHOLD].copy()

print(f"{len(edges)} pairs -> {len(strong_edges)} pairs with |r| >= {THRESHOLD}")
print(f"  positive correlations: {(strong_edges['weight'] > 0).sum()}")
print(f"  negative correlations: {(strong_edges['weight'] < 0).sum()}")
strong_edges.sort_values("weight").head()

### ✋ Exercise 1

Change `THRESHOLD` to 0.5, 0.6, 0.8 and 0.9, re-running from here to the degree table in
section 5, and record the number of nodes, the number of edges and the number of connected
components each time.

Then answer the question that matters: **at which threshold does the picture stop being
informative, and in which direction does it fail first?** Write one sentence you would be
willing to put in a methods section justifying your final choice — bearing in mind that
"the one that gave the prettiest network" is exactly the answer a reviewer is looking
for.

## 5. Building the co-abundance network

`nx.from_pandas_edgelist` does the whole conversion in one call: it reads the `source` and
`target` columns as node identifiers, creates the nodes it has not seen before, and — with
`edge_attr="weight"` — carries the correlation coefficient onto each edge. Keep that
attribute. An unweighted graph forgets whether an edge was r = 0.71 or r = 0.98, and every
weighted layout, every filter and every Cytoscape edge-width mapping needs it back.

The nodes are UniProt protein-group accessions, which nobody can read. Gene symbols are
kinder, so we attach them as a **node attribute** rather than renaming the nodes: the
accession stays the unambiguous key that joins to the annotation and to the published
results, and the symbol is what we print on the figure. Renaming nodes to gene symbols is
tempting and costs you the join — several protein groups share a symbol, and some have
none at all.

> 🧬 Recall what this graph asserts and what it does not. An edge says two proteins rise
> and fall together across these 45 septic patients. It is a hypothesis about shared
> regulation, a shared tissue of origin or a shared upstream driver. It is **not** evidence
> of a physical interaction, and it is not evidence of a common pathway.

In [ ]:
import networkx as nx

In [ ]:
# nx.from_pandas_edgelist builds the graph directly from the data frame;
# edge_attr="weight" carries the correlation coefficient onto each edge
G = nx.from_pandas_edgelist(
    strong_edges, source="source", target="target", edge_attr="weight"
)

# nodes are UniProt protein group identifiers; gene symbols are easier to read,
# so we attach them as a node attribute (falling back to the identifier when a
# protein group has no gene symbol)
gene_of = annotation["genes"].fillna(annotation.index.to_series()).to_dict()
nx.set_node_attributes(G, {n: gene_of.get(n, n) for n in G.nodes()}, name="gene")

labels = nx.get_node_attributes(G, "gene")

print(G)
print("example nodes:", [labels[n] for n in list(G.nodes())[:8]])

**How many nodes?** Note that this is smaller than the 150 proteins we started with:
a protein with no correlation above the threshold has no edges, so
`from_pandas_edgelist` never creates it. Isolated nodes vanish silently, which is usually
what you want and occasionally a surprise.

In [ ]:
print(G.number_of_nodes())

**How many edges?** Together with the node count this gives you the density — the fraction
of possible pairs that are actually connected — which is the single most useful number for
comparing two networks built the same way.

In [ ]:
print(G.number_of_edges())

**The degree of each node** is how many neighbours it has. We build it as a pandas Series,
add the gene symbols and sort, which is already a first ranking of the network. The mean
degree and the number of connected components describe the shape: a single giant component
means one interconnected block, while many small components mean the threshold has broken
the network into isolated pairs and triangles.

In [ ]:
degree_table = (
    pd.Series(dict(G.degree()), name="degree")
    .rename_axis("protein_group")
    .to_frame()
)
degree_table["gene"] = [gene_of.get(p, p) for p in degree_table.index]
degree_table = degree_table.sort_values("degree", ascending=False)

print("mean degree:", round(degree_table["degree"].mean(), 2))
print("connected components:", nx.number_connected_components(G))
degree_table.head(10)

## 6. Visualising the network

A **layout** is an algorithm that assigns each node an (x, y) position on the page. That is
all it is. It carries no biological meaning, it is often randomised, and the same network
drawn with two layouts can look like two different results — so never read a claim into the
positions alone, and never say "protein X sits at the centre" when you mean "protein X has
high degree".

| Layout | How it places nodes | Good for |
|---|---|---|
| **spring** (Fruchterman–Reingold) | physical simulation: edges pull, nodes repel | the default; shows clusters and hubs |
| **circular** | everything on a circle, in node order | comparing edge patterns without layout bias |
| **kamada-kawai** | minimises the difference between graph distance and page distance | small, well-connected networks |

Two practical notes about the code below. `spring_layout` takes a `seed` — set it, or your
figure changes every time you run the cell, which makes a notebook irreproducible in the
most avoidable way possible. And `kamada_kawai_layout` interprets `weight` as a *distance*,
so a negative correlation would be a negative distance; we pass `weight=None` to make it
ignore the sign.

In [ ]:
pos_spring = nx.spring_layout(G, seed=42)
pos_circular = nx.circular_layout(G)
# some layouts interpret "weight" as a distance and cannot handle negative
# values, so we tell kamada_kawai to ignore the correlation sign
pos_kk = nx.kamada_kawai_layout(G, weight=None)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, (name, pos) in zip(
    axes,
    [("spring", pos_spring), ("circular", pos_circular), ("kamada-kawai", pos_kk)],
):
    nx.draw_networkx(
        G,
        pos=pos,
        ax=ax,
        with_labels=False,
        node_size=60,
        node_color="#4878A8",
        edge_color="#CCCCCC",
        width=0.6,
    )
    ax.set_title(f"{name} layout")
    ax.axis("off")
fig.tight_layout()
plt.show()

**Adding the labels.** We draw edges, nodes and labels as three separate calls rather than
with the one-shot `nx.draw_networkx`, because that is what gives us control over each layer
— and it is the same pattern you would use to highlight a subset of nodes. The labels come
from the `gene` node attribute we set in section 5.

With this many nodes the labels will overlap. That is not a bug in your code; it is the
honest limit of matplotlib for network figures, and it is the reason the next hour is spent
in Cytoscape.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
nx.draw_networkx_edges(G, pos=pos_spring, ax=ax, edge_color="#DDDDDD", width=0.6)
nx.draw_networkx_nodes(G, pos=pos_spring, ax=ax, node_size=120, node_color="#4878A8")
# labels comes from the "gene" node attribute we set above
nx.draw_networkx_labels(G, pos=pos_spring, labels=labels, ax=ax, font_size=7)
ax.set_title(f"Serum protein co-abundance network (|r| >= {THRESHOLD}, n = 45 patients)")
ax.axis("off")
fig.tight_layout()
plt.show()

### Colour should encode something

An arbitrary palette is decoration; a mapped palette is data. Here we make both channels
carry information: **node colour and size** encode degree, and **edge colour** encodes the
sign of the correlation — red for proteins that move together, blue for proteins that move
in opposite directions — while **edge width** encodes |r|.

Note that the edge colouring is a two-way discrete mapping on the sign, and the node
colouring is a continuous mapping on a count. Those are the same two mapping types you will
set up in Cytoscape in section 9; the vocabulary transfers directly. See the
[NetworkX colour example](https://networkx.org/documentation/stable/auto_examples/drawing/plot_labels_and_colors.html)
for more of the same.

> ⚙️ One warning that applies to both tools: a **signed** quantity such as a fold change or
> a correlation needs a **diverging** palette centred on zero. A sequential palette hides
> the direction of the change, which is normally the whole point of the figure.

In [ ]:
node_degrees = [G.degree(n) for n in G.nodes()]
edge_colors = [
    "#C0392B" if G.edges[e]["weight"] > 0 else "#2471A3" for e in G.edges()
]
edge_widths = [2.5 * abs(G.edges[e]["weight"]) for e in G.edges()]

fig, ax = plt.subplots(figsize=(12, 10))
nx.draw_networkx_edges(
    G, pos=pos_spring, ax=ax, edge_color=edge_colors, width=edge_widths, alpha=0.6
)
nodes = nx.draw_networkx_nodes(
    G,
    pos=pos_spring,
    ax=ax,
    node_size=[40 + 25 * d for d in node_degrees],
    node_color=node_degrees,
    cmap=plt.cm.viridis,
)
nx.draw_networkx_labels(G, pos=pos_spring, labels=labels, ax=ax, font_size=7)
fig.colorbar(nodes, ax=ax, label="degree", shrink=0.6)
ax.set_title("Node colour and size = degree; edge colour = sign of the correlation")
ax.axis("off")
fig.tight_layout()
plt.show()

### ✋ Exercise 2

Work in new cells below (Colab: **+ Code**; JupyterLab: **Esc**, then **b**).

Rebuild the correlation matrix in section 3 with `method="spearman"` and run through to
this figure again. How many edges survive the same threshold, and — more interesting — how
much do the *hubs* change?

Then find one protein pair that is a strong edge under Pearson but not under Spearman, and
plot the two proteins against each other across the 45 patients. You are looking for a
single extreme patient carrying the correlation on their own. If you find one, you have
just learned why rank correlation is the safer default here.

## 7. Central nodes

Centrality ranks the nodes of a network by their number of connections, their position in
the topology, or the flow of information through them. NetworkX implements
[a great many measures](https://networkx.org/documentation/stable/reference/algorithms/centrality.html);
three are enough for most purposes.

| Measure | What it counts | High value means |
|---|---|---|
| **degree** | neighbours, divided by the maximum possible | the protein correlates with many others |
| **betweenness** | how often the node lies on a shortest path between two others | the protein bridges otherwise separate parts of the network |
| **closeness** | how short the paths to all other nodes are, on average | the protein sits in the well-connected core |

They disagree, and the disagreements are the interesting part: a node with high degree but
low betweenness sits inside one dense cluster, while a node with modest degree and high
betweenness is the only link between two clusters.

In [ ]:
centrality = pd.DataFrame(
    {
        "degree": nx.degree_centrality(G),
        "betweenness": nx.betweenness_centrality(G),
        "closeness": nx.closeness_centrality(G),
    }
).rename_axis("protein_group")
centrality["gene"] = [gene_of.get(p, p) for p in centrality.index]

print("Top 10 by betweenness centrality")
centrality.sort_values("betweenness", ascending=False).head(10).round(3)

### ⚠️ Hubs are suspicious as well as interesting

In a network of physical interactions, a node with high betweenness really can be a
bottleneck: a signal has to pass through it. A co-abundance network has no flow at all, so
the interpretation is much weaker. Read the ranking with these five caveats in mind — and
in that order, because the first two account for most of what you are looking at.

- **Abundance masquerades as centrality.** A protein that is highly abundant is measured
  precisely in every patient, and a precisely measured protein correlates with more things
  than a noisy one does. In serum this is not a subtle effect: albumin, the
  immunoglobulins, the apolipoproteins and the complement components dominate the dynamic
  range, and they are exactly the proteins that survived our complete-cases filter. If your
  top hubs are the proteins a clinician could have named before the experiment, you have
  probably measured the dynamic range of plasma rather than the biology of sepsis.
- **A single upstream driver produces a hub-shaped cluster.** If one inflammatory signal
  moves twenty proteins, all twenty correlate with each other and form a clique. The most
  central of them is not the cause; it is the best-measured member of the group.
- **Degree depends on the threshold.** Move the cut-off from 0.7 to 0.6 and the hubs
  change. A conclusion that survives only at one threshold is not a conclusion, which is
  what Exercise 1 was for.
- **Nothing travels along these edges.** A "shortest path" between two proteins is a
  property of our correlation matrix, not a route taken by molecules. Betweenness describes
  the shape of our table.
- **We used a subset.** Centrality was computed on 150 complete, highly variable proteins.
  A protein absent from that subset cannot be central, by construction.

Used properly, centrality is a way of **prioritising candidates for follow-up** — it points
at which five proteins to look up in STRING and read about tonight. It is not evidence in
itself, and a hub is not a finding until something outside this matrix agrees with it.

### ✋ Exercise 3

Work in new cells below (Colab: **+ Code**; JupyterLab: **Esc**, then **b**).

Take the top five proteins by degree and the top five by betweenness. For each, find its
mean log₂ intensity in `log_intensities` and its rank among all 1 458 protein groups by
abundance.

If the hubs are drawn from the top of the abundance distribution, what would you have to do
to the analysis to convince yourself that a hub is biological rather than technical? Name
one change to the preprocessing and one external source of evidence, and say what each one
would and would not rule out.

## 8. Node attributes worth carrying into Cytoscape

A network on its own is only topology, and a topology-only figure is rarely worth the page
it occupies. What makes a network figure informative is what is *attached* to the nodes —
the colour, the size and the border are where the biology goes. So we assemble a proper
node table before exporting anything:

| Attribute | Where it comes from | What it is for in the figure |
|---|---|---|
| `gene` | the annotation table | the node label |
| `degree`, `betweenness` | section 7 | node size, or a second visual channel |
| `is_dep_CRKP_vs_KP` | the paper's results | border width: which nodes the paper already flagged |
| `log2fc_CRKP_vs_KP`, `log2fc_CRKP_vs_Con` | the paper's results | node fill colour, on a diverging scale |

The published fold changes are what turn this from a structural picture into a biological
one: they let you ask whether the proteins the paper found differential are scattered
through the network or concentrated in one part of it — a question neither the network nor
the differential list can answer on its own.

> ⚙️ Note the handling of "not reported". Proteins the paper did not list for a comparison
> have no fold change, and we store `0.0` for them **with the flag alongside**. Zero and
> unknown are different statements, and a continuous colour mapping cannot tell them apart
> — the flag is what stops a white node from being read as "measured and unchanged".

In [ ]:
deps_all = pd.read_csv(PUBLISHED_DEPS, sep="\t")

dep_crkp_vs_kp = deps_all[deps_all["comparison"] == "CRKP_vs_KP"].set_index("protein_group")
dep_crkp_vs_con = deps_all[deps_all["comparison"] == "CRKP_vs_Con"].set_index("protein_group")

node_table = pd.DataFrame(index=pd.Index(sorted(G.nodes()), name="protein_group"))
node_table["gene"] = [gene_of.get(p, p) for p in node_table.index]
node_table["description"] = annotation["description"].reindex(node_table.index)
node_table["degree"] = [G.degree(p) for p in node_table.index]
node_table["betweenness"] = centrality["betweenness"].reindex(node_table.index).round(4)

# stored as 1/0 rather than True/False: both GraphML and Cytoscape handle
# integers unambiguously, and a discrete mapping on 1/0 is easy to set up
node_table["is_dep_CRKP_vs_KP"] = node_table.index.isin(dep_crkp_vs_kp.index).astype(int)
node_table["log2fc_CRKP_vs_KP"] = (
    dep_crkp_vs_kp["log2_fold_change"].reindex(node_table.index).fillna(0.0).round(3)
)
node_table["log2fc_CRKP_vs_Con"] = (
    dep_crkp_vs_con["log2_fold_change"].reindex(node_table.index).fillna(0.0).round(3)
)

# copy the attributes onto the graph itself, so that they are written to GraphML
for column in node_table.columns:
    nx.set_node_attributes(G, node_table[column].to_dict(), name=column)

print(f"{node_table['is_dep_CRKP_vs_KP'].sum()} of {len(node_table)} network proteins are "
      "published CRKP vs KP differential proteins")
node_table.sort_values("degree", ascending=False).head(10)

## 9. Exporting the network to Cytoscape

### Why leave Python at all?

matplotlib drew the network; it did not draw it well, and it will not draw it well however
long you spend on it. Three things push a network figure out of the notebook and into
[Cytoscape](https://cytoscape.org/) (Shannon *et al.*, 2003):

- **Layout you can control.** Force-directed layouts need nudging — you want to pull two
  labels apart, drag a cluster out of the middle, and then keep exactly that arrangement.
  Cytoscape saves the positions in the session, so the figure is reproducible rather than
  redrawn.
- **Rendering that survives a journal.** Vector output, real font control, edge bundling,
  legends. A 300-dpi PNG of a matplotlib network is not a publication figure.
- **Something you can hand over.** A colleague can open your session or your tables, click
  a node, and see its attributes. They cannot do anything with a screenshot.

### The two tables are the interchange format

Cytoscape does not want your Python objects; it wants tabular files, and the pair below is
the lingua franca of network exchange — every tool from Gephi to igraph to R's
`RCy3` reads the same two shapes.

| Table | One row per | Required columns | Everything else |
|---|---|---|---|
| **edge table** | protein pair | `source`, `target` — the node keys | edge attributes: `pearson_r`, `abs_r`, `sign` for width and colour |
| **node table** | protein | a key column matching `source`/`target` (here `protein_group`) | node attributes: `gene`, `degree`, `betweenness`, the fold changes, the DEP flag |

The **key** is the load-bearing part. Both tables must identify a node by the same string —
which is why we kept UniProt accessions as node identifiers and gene symbols as an
attribute back in section 5. Join on a display label and the import will silently drop
every protein group whose symbol is missing or shared.

We write both routes, because they fail differently:

1. **GraphML** (`.graphml`) — one file containing the topology, the edge weights and every
   node attribute. `nx.write_graphml` produces it and Cytoscape imports it in a single
   step. Cytoscape also reads GML (`nx.write_gml`), but GML handles attributes less
   cleanly.
2. **An edge table plus a node table** as two CSV files. More clicking, but this route
   always works, it opens in Excel or R, and it is the form you would deposit as
   supplementary material next to a paper.

In [ ]:
OUT_DIR = Path("cytoscape_export")
OUT_DIR.mkdir(exist_ok=True)

# 1. GraphML: topology + edge weights + every node attribute in a single file
graphml_file = OUT_DIR / "coabundance_network.graphml"
nx.write_graphml(G, graphml_file)

# 2a. edge table
edge_table = strong_edges.rename(columns={"weight": "pearson_r"}).copy()
edge_table["source_gene"] = [gene_of.get(p, p) for p in edge_table["source"]]
edge_table["target_gene"] = [gene_of.get(p, p) for p in edge_table["target"]]
edge_table["abs_r"] = edge_table["pearson_r"].abs().round(3)
edge_table["sign"] = np.where(edge_table["pearson_r"] > 0, "positive", "negative")
edge_table["pearson_r"] = edge_table["pearson_r"].round(3)
edge_table = edge_table[
    ["source", "target", "source_gene", "target_gene", "pearson_r", "abs_r", "sign"]
]
edge_file = OUT_DIR / "coabundance_edges.csv"
edge_table.to_csv(edge_file, index=False)

# 2b. node attribute table
node_file = OUT_DIR / "coabundance_nodes.csv"
node_table.to_csv(node_file)

for f in [graphml_file, edge_file, node_file]:
    print(f"{f}  ({f.stat().st_size / 1024:.1f} kB)")

edge_table.head()

### How to load these files in Cytoscape

Start Cytoscape (3.10 or newer) and then, depending on which file you want to use:

**Route 1 — the GraphML file (simplest, one step)**

1. `File -> Import -> Network from File...` and choose `coabundance_network.graphml`.
2. The network appears with every node attribute already in the node table. Nothing else to import.

**Route 2 — the two CSV files**

1. `File -> Import -> Network from File...` and choose `coabundance_edges.csv`.
   In the import dialog set the column meanings: `source` = **Source Node**, `target` = **Target Node**, and `pearson_r`, `abs_r`, `sign` = **Edge Attribute**. Click OK.
2. `File -> Import -> Table from File...` and choose `coabundance_nodes.csv`.
   Set **Key Column for Network** to `shared name` and the key column of the file to `protein_group`, so the rows are matched to the nodes that already exist. The remaining columns are imported as node attributes.

**Then style the network** in the `Style` panel (left-hand side, `Style` tab):

| Visual property | Column | Mapping type | Suggested setting |
| --- | --- | --- | --- |
| **Label** | `gene` | Passthrough Mapping | gene symbols as node labels |
| **Fill Colour** | `log2fc_CRKP_vs_KP` | Continuous Mapping | diverging palette, blue - white - red, centred on 0 |
| **Size** | `degree` | Continuous Mapping | e.g. 20 for the lowest degree, 70 for the highest |
| **Border Width** | `is_dep_CRKP_vs_KP` | Discrete Mapping | 1 -> thick border (published differential protein), 0 -> thin |
| **Edge Stroke Colour** | `sign` | Discrete Mapping | red = positive, blue = negative |
| **Edge Width** | `abs_r` | Continuous Mapping | thicker for stronger correlations |

A few practical points:

- A fold change is a signed quantity, so it needs a **diverging** palette centred on zero. A sequential palette (light to dark) hides the direction of the change, which is usually the whole point of the figure.
- Choose the layout in `Layout -> Prefuse Force Directed Layout` (or `yFiles Organic`, if installed). If you want the strong correlations to pull nodes closer together, set the edge weight column to `abs_r` in the layout settings.
- `Layout -> Apply Preferred Layout` after importing; Cytoscape's default grid layout tells you nothing.
- Export the figure with `File -> Export -> Network to Image...` (PDF or SVG for a paper, not PNG).

### Driving Cytoscape from Python

If Cytoscape is running on the same machine, you can skip the files entirely and control it from a notebook with **py4cytoscape**, which talks to the CyREST interface on `localhost:1234`. The block below is deliberately left as text rather than as a runnable cell: `py4cytoscape` is not installed in the course environment, and there is nothing for it to talk to on Colab. Read it as the shape of a reproducible pipeline you would set up locally.

```python
# pip install py4cytoscape   (not installed in this course environment)
import py4cytoscape as p4c

p4c.cytoscape_ping()                      # check that Cytoscape is listening
p4c.create_network_from_networkx(G, title="Serum co-abundance", collection="Course")
p4c.set_node_label_mapping("gene")
p4c.set_node_color_mapping(
    "log2fc_CRKP_vs_KP", [-2, 0, 2], ["#2471A3", "#FFFFFF", "#C0392B"]
)
p4c.set_node_size_mapping("degree", [1, 30], [20, 70])
p4c.layout_network("force-directed")
```

Notice that each call is one of the mappings from the table above — label passthrough, a diverging continuous colour mapping, a continuous size mapping, a layout. Scripting them is what makes a figure regenerable after a reviewer asks you to change the threshold. But it only works while a local Cytoscape is open. **The file route always works**, so export the GraphML and the CSVs even when you use py4cytoscape.

## 🔗 Where this goes next

We turned a 1 458 × 45 intensity matrix into a network of correlated serum proteins, ranked
its nodes, annotated them with the paper's differential proteins, and wrote the two tables
that any network tool can read. At **11:00 we open those tables in Cytoscape** — see
[`material/cytoscape.md`](https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/material/cytoscape.md) for the setup — and do the part that
Python does badly: style the network, lay it out by hand, and export something a journal
will accept. Bring `coabundance_network.graphml` and the two CSVs with you; if the export
cell did not run, run it now.

**Multi-omics II — Networks and pathways** takes the same construction and
does it across two omics layers at once, correlating proteins with metabolites over the
same 45 patients. Everything you have just learned about thresholds and hubs applies there
with more force, because a cross-omics edge is even easier to over-read than a
protein–protein one — and there the payoff is concrete: it is how the published analysis
reached the methionine cycle through **MAT2B**.

## 📚 Further reading

- [NetworkX documentation](https://networkx.org/documentation/stable/) — the
  [centrality reference](https://networkx.org/documentation/stable/reference/algorithms/centrality.html)
  and the [drawing examples](https://networkx.org/documentation/stable/auto_examples/index.html)
  are the two pages you will keep open.
- Shannon P *et al.* (2003) *Cytoscape: a software environment for integrated models of
  biomolecular interaction networks.* Genome Res 13:2498–2504. — the paper behind the tool
  we use at 11:00. [cytoscape.org](https://cytoscape.org/)
- [py4cytoscape](https://py4cytoscape.readthedocs.io/) — scripting Cytoscape from Python,
  for when the figure has to be regenerable.
- Weiss S *et al.* (2016) *Correlation detection strategies in microbial data sets vary
  widely in sensitivity and precision.* ISME J 10:1669–1681. — a hard-nosed benchmark of
  correlation-network methods; the failure modes it documents are the ones in section 7.
- Zhang B & Horvath S (2005) *A general framework for weighted gene co-expression network
  analysis.* Stat Appl Genet Mol Biol 4:17. — the soft-thresholding alternative to picking
  one cut-off, and worth reading precisely because it disagrees with what we did here.
- Gysi DM & Nowick K (2020) *Construction, comparison and evolution of networks in life
  sciences and other disciplines.* J R Soc Interface 17:20190610. — a survey of how
  co-expression and co-abundance networks are built, and how they go wrong.